In [1]:
import argparse
import os
import subprocess
from pathlib import Path
from ase.visualize import view
from readqe import read_qe_hubbard_out
from ase.io import read

In [2]:
metals = ['VN']
adsorbates = ['Li2S','Li2S2','Li2S4','Li2S6','Li2S8','S8']

for metal in metals:
    for ads in adsorbates:
        traj_path = Path(f'./final_xyz/{metal}_{ads}_final.xyz')
        if not traj_path.exists():
            print(f'Skipping missing: {traj_path}')
            continue

        combi = read(str(traj_path), index='-1')
        view(combi)

In [ ]:
metals = ['VN']
adsorbates = ['Li2S','Li2S2','Li2S4','Li2S6','Li2S8','S8']

for metal in metals:
    for ads in adsorbates:
        traj_path = Path(f'{metal}_{ads}_combi_relax_u/{metal}_{ads}_finals.traj')
        if not traj_path.exists():
            print(f'Skipping missing: {traj_path}')
            continue

        frames = read(str(traj_path), index=':')
        if not frames:
            continue

        print(f'\n{metal}_{ads}: {len(frames)} frame(s) in {traj_path}')
        view(frames)


TiN_S8: 3 frame(s) in TiN_S8_combi_relax_u/TiN_S8_finals.traj

VN_S8: 7 frame(s) in VN_S8_combi_relax_u/VN_S8_finals.traj


In [5]:
from pathlib import Path
from ase.io.trajectory import Trajectory

versions = ["u1", "u2", "u3", "u4"]
all_frames = []

for relax_dir in sorted(Path(".").glob("*_combi_relax_u")):
    base = relax_dir.name.replace("_combi_relax_u", "")
    frames = []

    for v in versions:
        out_file = relax_dir / f"{base}_combi_{v}.out"
        if not out_file.exists():
            continue

        try:
            traj_data = read_qe_hubbard_out(str(out_file), index=":")
        except ValueError as e:
            # Skip incomplete/bad outputs with no parsed geometry snapshots.
            print(f"Skipping {out_file}: {e}")
            continue

        if isinstance(traj_data, list):
            if not traj_data:
                continue
            atoms = traj_data[-1].copy()  # keep final structure from that run
        else:
            atoms = traj_data.copy()

        atoms.info["source_out"] = out_file.name
        atoms.info["version"] = v
        try:
            atoms.info["energy_eV"] = float(atoms.get_potential_energy())
        except Exception:
            pass
        frames.append(atoms)

    if not frames:
        continue

    system_traj = relax_dir / f"{base}_finals.traj"
    with Trajectory(str(system_traj), "w") as t:
        for a in frames:
            t.write(a)

    all_frames.extend(frames)
    print(f"Wrote {len(frames)} frames -> {system_traj}")

if all_frames:
    combined_traj = Path("combi_u1_to_u4_finals_all_systems.traj")
    with Trajectory(str(combined_traj), "w") as t:
        for a in all_frames:
            t.write(a)
    print(f"Wrote {len(all_frames)} total frames -> {combined_traj}")
else:
    print("No u1-u4 .out files found.")

Wrote 1 frames -> TiN_Li2S2_combi_relax_u/TiN_Li2S2_finals.traj
Wrote 3 frames -> TiN_Li2S4_combi_relax_u/TiN_Li2S4_finals.traj
Wrote 4 frames -> TiN_Li2S6_combi_relax_u/TiN_Li2S6_finals.traj
Skipping TiN_Li2S8_combi_relax_u/TiN_Li2S8_combi_u4.out: No geometry snapshots found in TiN_Li2S8_combi_relax_u/TiN_Li2S8_combi_u4.out
Wrote 3 frames -> TiN_Li2S8_combi_relax_u/TiN_Li2S8_finals.traj
Wrote 1 frames -> TiN_Li2S_combi_relax_u/TiN_Li2S_finals.traj
Wrote 3 frames -> TiN_S8_combi_relax_u/TiN_S8_finals.traj
Wrote 1 frames -> VN_Li2S2_combi_relax_u/VN_Li2S2_finals.traj
Wrote 1 frames -> VN_Li2S4_combi_relax_u/VN_Li2S4_finals.traj
Wrote 4 frames -> VN_Li2S6_combi_relax_u/VN_Li2S6_finals.traj
Wrote 4 frames -> VN_Li2S8_combi_relax_u/VN_Li2S8_finals.traj
Wrote 1 frames -> VN_Li2S_combi_relax_u/VN_Li2S_finals.traj
Wrote 4 frames -> VN_S8_combi_relax_u/VN_S8_finals.traj
Wrote 30 total frames -> combi_u1_to_u4_finals_all_systems.traj


In [7]:
from pathlib import Path
import re
import numpy as np
from ase.io import read
from ase.io.trajectory import Trajectory

def as_list(frames):
    if isinstance(frames, list):
        return frames
    return [frames]

def geom_signature(atoms, decimals=5):
    return (
        tuple(atoms.get_chemical_symbols()),
        tuple(np.round(atoms.positions.reshape(-1), decimals)),
        tuple(np.round(atoms.cell.array.reshape(-1), decimals)),
        tuple(bool(x) for x in atoms.pbc),
    )

combined_traj = Path("combi_u1_to_u4_finals_all_systems.traj")
appended_total = 0
skipped_total = 0

combined_meta = set()
combined_geom = set()
if combined_traj.exists():
    existing_combined = as_list(read(str(combined_traj), index=":"))
    for a in existing_combined:
        combined_meta.add((a.info.get("source_out"), a.info.get("version")))
        combined_geom.add(geom_signature(a))

for relax_dir in sorted(Path(".").glob("*_combi_relax_u")):
    base = relax_dir.name.replace("_combi_relax_u", "")
    system_traj = relax_dir / f"{base}_finals.traj"

    out_files = list(relax_dir.glob(f"{base}_combi_u*.out"))
    out_files += list(Path("outputs").glob(f"{base}_combi_u*.out"))
    out_files = sorted(set(out_files), key=lambda p: p.name)

    if not out_files:
        continue

    existing_meta = set()
    existing_geom = set()
    if system_traj.exists():
        existing_system = as_list(read(str(system_traj), index=":"))
        for a in existing_system:
            existing_meta.add((a.info.get("source_out"), a.info.get("version")))
            existing_geom.add(geom_signature(a))

    frames_to_append = []
    skipped_here = 0

    for out_file in out_files:
        m = re.search(r"_combi_(u\d+)\.out$", out_file.name)
        version = m.group(1) if m else "unknown"

        try:
            traj_data = read_qe_hubbard_out(str(out_file), index=":")
        except ValueError as e:
            print(f"Skipping {out_file}: {e}")
            continue

        if isinstance(traj_data, list):
            if not traj_data:
                continue
            atoms = traj_data[-1].copy()
        else:
            atoms = traj_data.copy()

        atoms.info["source_out"] = out_file.name
        atoms.info["version"] = version
        try:
            atoms.info["energy_eV"] = float(atoms.get_potential_energy())
        except Exception:
            pass

        meta_key = (atoms.info.get("source_out"), atoms.info.get("version"))
        geom_key = geom_signature(atoms)

        is_dup_system = (meta_key in existing_meta) or (geom_key in existing_geom)
        is_dup_combined = (meta_key in combined_meta) or (geom_key in combined_geom)
        if is_dup_system or is_dup_combined:
            skipped_here += 1
            continue

        frames_to_append.append(atoms)
        existing_meta.add(meta_key)
        existing_geom.add(geom_key)
        combined_meta.add(meta_key)
        combined_geom.add(geom_key)

    if not frames_to_append:
        if skipped_here > 0:
            print(f"No new frames for {base} (skipped {skipped_here} duplicate(s)).")
        continue

    with Trajectory(str(system_traj), "a") as t:
        for atoms in frames_to_append:
            t.write(atoms)

    with Trajectory(str(combined_traj), "a") as t:
        for atoms in frames_to_append:
            t.write(atoms)

    appended_total += len(frames_to_append)
    skipped_total += skipped_here
    print(
        f"Appended {len(frames_to_append)} new frame(s) -> {system_traj} "
        f"(skipped {skipped_here} duplicate(s))"
    )

print(f"Appended {appended_total} new frame(s) total -> {combined_traj}")
print(f"Skipped {skipped_total} duplicate frame(s) total")

No new frames for TiN_Li2S2 (skipped 1 duplicate(s)).
No new frames for TiN_Li2S4 (skipped 3 duplicate(s)).
No new frames for TiN_Li2S6 (skipped 5 duplicate(s)).
Skipping outputs/TiN_Li2S8_combi_u4.out: No geometry snapshots found in outputs/TiN_Li2S8_combi_u4.out
No new frames for TiN_Li2S8 (skipped 4 duplicate(s)).
No new frames for TiN_Li2S (skipped 1 duplicate(s)).
No new frames for TiN_S8 (skipped 3 duplicate(s)).
No new frames for VN_Li2S2 (skipped 1 duplicate(s)).
No new frames for VN_Li2S4 (skipped 1 duplicate(s)).
Appended 1 new frame(s) -> VN_Li2S6_combi_relax_u/VN_Li2S6_finals.traj (skipped 6 duplicate(s))
No new frames for VN_Li2S8 (skipped 6 duplicate(s)).
No new frames for VN_Li2S (skipped 1 duplicate(s)).
Appended 1 new frame(s) -> VN_S8_combi_relax_u/VN_S8_finals.traj (skipped 6 duplicate(s))
Appended 2 new frame(s) total -> combi_u1_to_u4_finals_all_systems.traj
Skipped 12 duplicate frame(s) total


In [6]:
import numpy as np
from pathlib import Path
from ase.io import read

base = "VN_Li2S6"
relax_dir = Path(f"{base}_combi_relax_u")
traj_path = relax_dir / f"{base}_finals.traj"

# Prefer the direct combi_u.out in the relax folder; fallback to outputs if needed.
out_candidates = [
    relax_dir / f"{base}_combi_u.out",
    Path("outputs") / f"{base}_combi_u.out",
]

out_path = None
for p in out_candidates:
    if p.exists():
        out_path = p
        break

if out_path is None:
    extra = sorted(Path("outputs").glob(f"{base}_combi_u*.out"))
    if extra:
        out_path = max(extra, key=lambda p: p.stat().st_mtime)
        print(f"Using fallback output file: {out_path}")
    else:
        raise FileNotFoundError(f"No {base}_combi_u*.out found in relax folder or outputs/")

if not traj_path.exists():
    raise FileNotFoundError(f"Missing trajectory file: {traj_path}")

out_data = read_qe_hubbard_out(str(out_path), index=":")
traj_data = read(str(traj_path), index=":")

if isinstance(out_data, list):
    if not out_data:
        raise ValueError(f"No snapshots parsed from {out_path}")
    out_last = out_data[-1]
else:
    out_last = out_data

if isinstance(traj_data, list):
    if not traj_data:
        raise ValueError(f"No frames in {traj_path}")
    traj_last = traj_data[-1]
else:
    traj_last = traj_data

same_natoms = len(out_last) == len(traj_last)
same_symbols = out_last.get_chemical_symbols() == traj_last.get_chemical_symbols()

if not same_natoms or not same_symbols:
    print(f"natoms same: {same_natoms}")
    print(f"symbols same: {same_symbols}")
    print("Result: NOT THE SAME")
else:
    pos_diff = out_last.positions - traj_last.positions
    rms = float(np.sqrt((pos_diff**2).mean()))
    max_abs = float(np.abs(pos_diff).max())
    cell_diff = float(np.abs(out_last.cell.array - traj_last.cell.array).max())
    pbc_same = np.array_equal(out_last.pbc, traj_last.pbc)

    tol_pos = 1.0e-4
    tol_cell = 1.0e-6
    same_geom = (max_abs < tol_pos) and (cell_diff < tol_cell) and pbc_same

    print(f"Compared: {out_path} (last) vs {traj_path} (last)")
    print(f"RMS position diff: {rms:.6e} Ang")
    print(f"Max position diff: {max_abs:.6e} Ang")
    print(f"Max cell diff:     {cell_diff:.6e} Ang")
    print(f"PBC identical:     {pbc_same}")
    print(f"Result: {'SAME' if same_geom else 'NOT THE SAME'}")

Compared: VN_Li2S6_combi_relax_u/VN_Li2S6_combi_u.out (last) vs VN_Li2S6_combi_relax_u/VN_Li2S6_finals.traj (last)
RMS position diff: 1.336618e-02 Ang
Max position diff: 1.397541e-01 Ang
Max cell diff:     0.000000e+00 Ang
PBC identical:     True
Result: NOT THE SAME


In [ ]:
file = ('/home/ameer_ubuntu/Git_projects/QE_2/u/VN_Li2S4_relax.out')
slab = read_qe_hubbard_out(file, index=':')
view(slab)

<Popen: returncode: None args: ['/home/ameer_ubuntu/miniforge3/envs/qe/bin/p...>

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/ameer_ubuntu/miniforge3/envs/qe/lib/python3.13/site-packages/ase/gui/pipe.py", line 34, in <module>
    main()
    ~~~~^^
  File "/home/ameer_ubuntu/miniforge3/envs/qe/lib/python3.13/site-packages/ase/gui/pipe.py", line 30, in main
    plt.show()
    ~~~~~~~~^^
  File "/home/ameer_ubuntu/miniforge3/envs/qe/lib/python3.13/site-packages/matplotlib/pyplot.py", line 613, in show
    return _get_backend_mod().show(*args, **kwargs)
           ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/home/ameer_ubuntu/miniforge3/envs/qe/lib/python3.13/site-packages/matplotlib_inline/backend_inline.py", line 90, in show
    display(
    ~~~~~~~^
        figure_manager.canvas.figure,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        metadata=_fetch_figure_metadata(figure_manager.canvas.figure),
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^